# 🕵️ Part 1 — The Code Auditor Agent
### Workshop 4 · Hands-on: Building Cooperative LLM Agent Workflows for Anti-pattern Detection, Code Smell & Technical Debt Resolution
*LLMA4SE Summer School 2026 · Karthik Shivashankar & Adela Nedisan Videsjorden*

---

**⏱ Time:** ~55 min &nbsp;·&nbsp; **Runtime needed:** free Google Colab **T4 GPU**

> ☝️ **Before anything else:** `Runtime → Change runtime type → T4 GPU → Save`

### What you will build in Part 1

By the end of this notebook you will have a working **Code Auditor Agent**: an LLM that *uses static-analysis tools* (radon, pylint, and **PyExamine** — a real research tool from MSR 2025) to find code smells in a Python module and explain them like a senior reviewer would.

```
                 ┌──────────────────────────────┐
   smelly        │       CODE AUDITOR AGENT      │      structured
   code.py ────► │  tools: radon · pylint ·      │ ───► findings
                 │         PyExamine             │      (JSON)
                 │  brain: local LLM (1.5B)      │
                 └──────────────────────────────┘
```

**The one idea to hold onto for the next 3 hours:**

> 🧠 An **agent** = an LLM + a **role** (system prompt) + **tools** it can call + a **structured output** contract. Nothing more magical than that.


## 1.1 · Check your hardware (1 min)

Free Colab gives you an NVIDIA T4 with ~15 GB of VRAM — plenty for a 1.5B-parameter code model in half precision.

In [ ]:
!nvidia-smi
import torch
print("\nPyTorch sees CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU, then re-run.")

## 1.2 · Install the toolbox (2 min)

| Package | What it is | Why an agent needs it |
|---|---|---|
| `transformers` + `accelerate` | Hugging Face model runtime | the agent's *brain* |
| `radon` | complexity & maintainability metrics | fast, deterministic *eyes* |
| `pylint` | classic Python linter | more deterministic *eyes* |
| `code-quality-analyzer` | **PyExamine** (Shivashankar & Martini, MSR 2025) — 49-metric smell detector | research-grade *eyes* 👀 |
| `ml_smells_detector` | **MLScent** (Shivashankar, CAIN 2025) — 76 ML-specific anti-pattern detectors | *eyes for ML code* 🤖 |

In [ ]:
%pip install -q transformers accelerate radon pylint code-quality-analyzer pytest
%pip install -q git+https://github.com/KarthikShivasankar/ml_smells_detector.git
print("✅ toolbox installed")

## 1.3 · Meet the patient 🤒

We need a codebase to operate on. `inventory.py` below is a small order-processing module that **works perfectly** — every test passes — yet it is riddled with classic smells:

mutable default argument · dead code · long parameter list · long method · deep nesting · magic numbers · duplicated logic.

**Working ≠ healthy.** That gap is exactly what *technical debt* is.

In [ ]:
%%writefile inventory.py
"""inventory.py -- Order processing for a small e-commerce shop.

This module works correctly (all tests pass!) but it is deliberately
full of code smells. Your agents will find and fix them.
"""

def helper_unused(x):          # SMELL: dead code -- never called anywhere
    return x * 2


class InventoryManager:
    """Manages stock and processes customer orders."""

    def __init__(self, items=[]):              # SMELL: mutable default argument
        self.items = {}
        for name, price, qty in items:
            self.items[name] = {"price": price, "qty": qty}
        self.log = []

    def add_item(self, name, price, qty, category, supplier, discount, taxable):
        # SMELL: long parameter list (7 params, most unused)
        self.items[name] = {"price": price, "qty": qty}
        return True

    def process_order(self, order):
        # SMELL: long method, deep nesting, magic numbers, duplication
        total = 0.0
        status = "ok"
        for name, qty in order:
            if name in self.items:
                if self.items[name]["qty"] >= qty:
                    if qty > 0:
                        price = self.items[name]["price"]
                        subtotal = price * qty
                        if subtotal > 100:                      # magic number
                            subtotal = subtotal - subtotal * 0.05   # magic number
                        if qty > 10:                            # magic number
                            subtotal = subtotal - subtotal * 0.02   # magic number
                        total = total + subtotal
                        self.items[name]["qty"] = self.items[name]["qty"] - qty
                        self.log.append("sold " + name)
                    else:
                        status = "invalid_qty"
                else:
                    status = "insufficient_stock"
            else:
                status = "unknown_item"
        total = total + total * 0.25            # magic number (VAT)
        return {"total": round(total, 2), "status": status}

    def refund_order(self, order):
        # SMELL: duplicated logic (mirror of process_order maths)
        total = 0.0
        for name, qty in order:
            if name in self.items:
                price = self.items[name]["price"]
                subtotal = price * qty
                if subtotal > 100:                              # magic number again
                    subtotal = subtotal - subtotal * 0.05
                if qty > 10:
                    subtotal = subtotal - subtotal * 0.02
                total = total + subtotal
                self.items[name]["qty"] = self.items[name]["qty"] + qty
        total = total + total * 0.25
        return {"total": round(total, 2), "status": "refunded"}

    def get_stock(self, name):
        if name in self.items:
            return self.items[name]["qty"]
        return 0


### The safety net 🥅

Before we let *any* AI touch this code, we pin down its behaviour with tests. These tests are the **contract**: a refactoring is only valid if they stay green.

> 💬 **Discuss (30 s with your neighbour):** why must the tests be written *before* the refactoring agent runs, not after?

In [ ]:
%%writefile test_inventory.py
"""test_inventory.py -- Behaviour-preserving safety net.

These tests define the PUBLIC CONTRACT of the module. Any refactoring
your agents perform MUST keep every one of these green.
"""
import pytest
from inventory import InventoryManager


@pytest.fixture
def mgr():
    return InventoryManager([("widget", 10.0, 100), ("gizmo", 25.0, 5)])


def test_simple_order(mgr):
    result = mgr.process_order([("widget", 2)])
    assert result["status"] == "ok"
    assert result["total"] == 25.0          # 20 + 25% VAT


def test_bulk_discount_applied(mgr):
    # 20 widgets = 200 -> -5% (>100) -> -2% (>10 units) -> +25% VAT
    result = mgr.process_order([("widget", 20)])
    assert result["total"] == 232.75


def test_stock_is_decremented(mgr):
    mgr.process_order([("widget", 2)])
    assert mgr.get_stock("widget") == 98


def test_insufficient_stock(mgr):
    result = mgr.process_order([("gizmo", 99)])
    assert result["status"] == "insufficient_stock"


def test_unknown_item(mgr):
    result = mgr.process_order([("nonexistent", 1)])
    assert result["status"] == "unknown_item"


def test_refund_restores_stock(mgr):
    mgr.process_order([("widget", 2)])
    mgr.refund_order([("widget", 2)])
    assert mgr.get_stock("widget") == 100


def test_no_shared_state_between_instances():
    a = InventoryManager()
    b = InventoryManager()
    a.items["x"] = {"price": 1, "qty": 1}
    assert "x" not in b.items or a.items is not b.items


In [ ]:
# The smelly code WORKS — that's the whole point:
!python -m pytest test_inventory.py -q

## 1.4 · Deterministic tools first — the agent's "eyes"

A core design rule of agentic software engineering:

> ⚖️ **Never make an LLM guess what a deterministic tool can measure.**
> Static analysers are fast, cheap, and never hallucinate. The LLM's job is *interpretation and action*, not *measurement*.

We wrap three analysers as plain Python functions. These become the agent's **tools**.

### Tool 1 — `radon`: cyclomatic complexity & maintainability

In [ ]:
import subprocess, json

def run_radon(path: str) -> str:
    """Cyclomatic complexity (CC) per function + Maintainability Index (MI)."""
    cc = subprocess.run(["radon", "cc", "-s", path], capture_output=True, text=True).stdout
    mi = subprocess.run(["radon", "mi", "-s", path], capture_output=True, text=True).stdout
    return f"CYCLOMATIC COMPLEXITY (A=best, F=worst):\n{cc}\nMAINTAINABILITY INDEX (100=best):\n{mi}"

print(run_radon("inventory.py"))

📊 **Read the output:** `process_order` should stand out. CC counts independent paths through a function — every `if` adds one. High CC = hard to test, hard to change.

### Tool 2 — `pylint`: rule-based linting

In [ ]:
def run_pylint(path: str, max_findings: int = 15) -> str:
    """Classic linter findings as compact text."""
    raw = subprocess.run(
        ["pylint", path, "--output-format=json", "--disable=C0114,C0115,C0116"],
        capture_output=True, text=True
    ).stdout
    try:
        issues = json.loads(raw)[:max_findings]
    except json.JSONDecodeError:
        return raw[:1500]
    return "\n".join(f"L{i['line']}: [{i['symbol']}] {i['message']}" for i in issues) or "no findings"

print(run_pylint("inventory.py"))

👀 Notice pylint catches the **mutable default argument** (`dangerous-default-value`) — a bug-in-waiting that radon's metrics are blind to. Different tools see different smells. That's why real auditors combine them.

### Tool 3 — **PyExamine** 🔬 (the research tool)

PyExamine analyses code at **three levels** — code smells, structural smells, architectural smells — across 49 metrics, and reached **91% recall** in its MSR 2025 evaluation. It's built by today's instructor, so complaints go straight to the source 😄

In [ ]:
import pathlib

def run_pyexamine(directory: str = ".") -> str:
    """PyExamine: multi-level smell detection (Shivashankar & Martini, MSR 2025).
    Findings are written to <output>.txt, so we run the CLI then read the report."""
    subprocess.run(
        ["analyze_code_quality", directory, "--type", "code",
         "--output", "pyexamine_report",
         "--ignore", "sample_data", ".config", "__pycache__"],
        capture_output=True, text=True, timeout=300,
    )
    report = pathlib.Path("pyexamine_report.txt")
    return report.read_text()[:3000] if report.exists() else "PyExamine produced no report"

print(run_pyexamine("."))

> 🧪 **Try it (2 min):** re-run with `--type structural` (edit the function or call `subprocess` directly). Which *new* smells appear that the code-level pass missed?

### Tool 4 — **MLScent** 🤖 (smells that only exist in ML code)

Classic tools are blind to a whole category of rot that lives *only* in machine-learning code: a missing random seed, a `== np.nan` comparison (always `False`!), a forgotten `optimizer.zero_grad()`. These don't crash — they silently make your model **unreproducible or subtly wrong**, which is arguably worse.

**MLScent** (Shivashankar, CAIN 2025) implements **76 AST-based detectors** across TensorFlow (13), PyTorch (12), Scikit-learn (9), Hugging Face (10), Pandas & NumPy (8 each) + 16 general ML smells.

First, a second patient — an ML training script that "trains fine":

In [ ]:
!mkdir -p ml_project

In [ ]:
%%writefile ml_project/train_model.py
"""train_model.py -- Churn-prediction training script.

It trains fine... but is it reproducible? Is it healthy ML code?
Your ML Auditor agent (powered by MLScent) will tell you.
"""
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def load_data(path):
    df = pd.read_csv(path)
    for i in range(len(df)):                      # pandas: unnecessary iteration
        if df["age"][i] == np.nan:                # numpy: NaN equality (always False!)
            df["age"][i] = 0                      # pandas: chain indexing
    return df


def train():
    df = load_data("churn.csv")
    X = df.drop("label", axis=1).values
    y = df["label"].values
    # sklearn: no feature scaling, no pipeline, no random_state
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

    model = nn.Sequential(nn.Linear(X.shape[1], 64), nn.ReLU(), nn.Linear(64, 2))
    opt = torch.optim.Adam(model.parameters(), lr=0.003)   # hardcoded hyperparams
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(100):                      # no early stopping, no checkpoints
        out = model(torch.tensor(X_train, dtype=torch.float32))
        loss = loss_fn(out, torch.tensor(y_train))
        loss.backward()                           # pytorch: missing opt.zero_grad()
        opt.step()

    preds = model(torch.tensor(X_test, dtype=torch.float32)).argmax(1).numpy()
    print("accuracy:", accuracy_score(y_test, preds))   # over-reliance on accuracy
    # no torch.manual_seed / np.random.seed anywhere -> unreproducible


if __name__ == "__main__":
    train()


> 😱 Before running the detector: **spot 3 smells yourself** (60 seconds, no scrolling back). The comments are turned off in your head, right?

Now wrap MLScent as a tool, exactly like the others:

In [ ]:
def run_mlscent(project_dir: str = "ml_project") -> str:
    """MLScent: 76 ML-specific anti-pattern detectors (Shivashankar, CAIN 2025).
    Findings land in output/analysis_report.txt, so we run the CLI then read it."""
    subprocess.run(["ml_smell_detector", "analyze", project_dir],
                   capture_output=True, text=True, timeout=300)
    report = pathlib.Path("output/analysis_report.txt")
    return report.read_text()[:3500] if report.exists() else "MLScent produced no report"

print(run_mlscent("ml_project"))

📊 **Read the output:** MLScent typically flags ~14 issues here — including the `== np.nan` bug (a *correctness* problem: that branch never executes) and the missing `zero_grad()` (gradients accumulate across epochs → wrong training). Also note each finding ships a **"How to fix"** — that's structured refactoring advice we'll feed to agents later.

## 1.5 · Load the agent's brain 🧠

We use **Qwen2.5-Coder-1.5B-Instruct** — small enough for a free T4, strong enough for code reasoning. The `llm()` helper below is the *only* interface every agent in this workshop will use. (~2 min to download.)

In [ ]:
# ============================================================
#  Load a small open-weights code LLM (fits free Colab T4 GPU)
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # ~3 GB in fp16
# Slower machine / CPU-only fallback:
# MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME} on {device} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
print("Model loaded ✔")


def llm(user_prompt: str, system_prompt: str = "You are a helpful assistant.",
        max_new_tokens: int = 1024, temperature: float = 0.2) -> str:
    """One call to our local LLM. Every agent in this workshop uses this."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()


In [ ]:
# Smoke test — is the brain alive?
print(llm("In one sentence: what is a code smell?"))

## 1.6 · Anatomy of an agent

Here is our minimal, framework-free agent. Production frameworks (LangGraph, AutoGen, CrewAI) add scheduling, persistence and tracing — but **this is the essential skeleton they all share**:

| Ingredient | In our class | In production frameworks |
|---|---|---|
| **Role** | `system_prompt` | "agent persona / instructions" |
| **Brain** | `llm()` | any model endpoint |
| **Tools** | Python callables | tool / function-calling schemas |
| **Output contract** | "reply ONLY in JSON" | structured output / schemas |


In [ ]:
class Agent:
    """Minimal agent: a role + a brain + tools + an output contract."""

    def __init__(self, name: str, system_prompt: str, tools: dict | None = None):
        self.name = name
        self.system_prompt = system_prompt
        self.tools = tools or {}          # {"tool_name": callable}

    def use_tools(self, *args) -> str:
        """Run every tool and concatenate the evidence."""
        report = []
        for tool_name, fn in self.tools.items():
            print(f"  🔧 {self.name} is running tool: {tool_name}")
            try:
                report.append(f"=== {tool_name} ===\n{fn(*args)}")
            except Exception as e:
                report.append(f"=== {tool_name} FAILED: {e} ===")
        return "\n\n".join(report)

    def think(self, prompt: str, **kw) -> str:
        return llm(prompt, system_prompt=self.system_prompt, **kw)

### The Code Auditor

Its workflow: **(1)** run all tools → **(2)** hand the raw evidence + source code to the LLM → **(3)** demand structured JSON findings.

Note the two prompt-engineering moves that make small models reliable:
1. **Evidence-grounding** — the LLM summarises tool output instead of inventing findings.
2. **A strict JSON schema in the prompt** — so downstream agents can *parse*, not *read*, the result.

In [ ]:
import re

AUDITOR_PROMPT = """You are a meticulous senior code reviewer.
You receive: (a) a Python source file, (b) evidence from static-analysis tools.
Identify the most important code smells / anti-patterns.

Reply ONLY with a JSON array (no prose, no markdown fences). Each element:
{"smell": "<short name>", "location": "<function/line>",
 "severity": "high|medium|low", "why": "<one sentence>",
 "fix": "<one-sentence refactoring suggestion>"}
List at most 6 findings, most severe first."""

def extract_json(text: str):
    """LLMs love to wrap JSON in chatter — dig the array/object out."""
    m = re.search(r"\[.*\]|\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError(f"No JSON found in: {text[:200]}")
    return json.loads(m.group(0))

auditor = Agent(
    name="Code Auditor",
    system_prompt=AUDITOR_PROMPT,
    tools={"radon": run_radon, "pylint": run_pylint},
)

def audit(path: str) -> list[dict]:
    evidence = auditor.use_tools(path)
    source = open(path).read()
    raw = auditor.think(
        f"SOURCE FILE ({path}):\n```python\n{source}\n```\n\n"
        f"TOOL EVIDENCE:\n{evidence}\n\nProduce the JSON findings now.",
        max_new_tokens=800, temperature=0.1,
    )
    return extract_json(raw)

findings = audit("inventory.py")
findings

In [ ]:
# Pretty-print the audit like a review dashboard
SEV = {"high": "🔴", "medium": "🟠", "low": "🟡"}
print(f"{'':2} {'SMELL':28} {'WHERE':22} WHY")
print("-" * 95)
for f in findings:
    print(f"{SEV.get(f.get('severity','low'),'⚪')} {f.get('smell','?'):28.28} "
          f"{str(f.get('location','?')):22.22} {f.get('why','')}")
    print(f"   ↳ fix: {f.get('fix','')}\n")

🎉 **You built your first agent.** Look at what just happened: deterministic tools did the measuring, the LLM did the interpreting, and a JSON contract made the result machine-readable — ready to hand to the *next* agent in Part 2.

## 1.7 · Same skeleton, different eyes: the **ML Auditor** 🤖

Here's the payoff of the agent abstraction: to review *machine-learning* code instead of business logic, we change exactly **two ingredients** — the role and the tools. The skeleton, the JSON contract, the `llm()` call: untouched.

Notice how the role prompt shifts the reviewer's *values*: reproducibility and silent-correctness bugs now outrank readability.

In [ ]:
ML_AUDITOR_PROMPT = """You are a senior ML engineer reviewing training code.
You receive: (a) an ML training script, (b) evidence from MLScent, a static
analyser with 76 ML-specific detectors.
Prioritise: (1) silent correctness bugs, (2) reproducibility,
(3) training hygiene (early stopping, checkpoints, eval mode), (4) style.

Reply ONLY with a JSON array (no prose). Each element:
{"smell": "<short name>", "impact": "correctness|reproducibility|hygiene|style",
 "severity": "high|medium|low", "why": "<one sentence>",
 "fix": "<one-sentence fix>"}
List at most 6 findings, most severe first."""

ml_auditor = Agent(
    name="ML Auditor",
    system_prompt=ML_AUDITOR_PROMPT,
    tools={"mlscent": run_mlscent},
)

def ml_audit(project_dir: str, main_file: str) -> list[dict]:
    evidence = ml_auditor.use_tools(project_dir)
    source = open(main_file).read()
    raw = ml_auditor.think(
        f"TRAINING SCRIPT ({main_file}):\n```python\n{source}\n```\n\n"
        f"MLSCENT EVIDENCE:\n{evidence}\n\nProduce the JSON findings now.",
        max_new_tokens=800, temperature=0.1,
    )
    return extract_json(raw)

ml_findings = ml_audit("ml_project", "ml_project/train_model.py")

IMPACT = {"correctness": "💥", "reproducibility": "🎲", "hygiene": "🧼", "style": "✏️"}
for f in ml_findings:
    print(f"{IMPACT.get(f.get('impact','style'),'•')} [{f.get('severity','?'):6}] "
          f"{f.get('smell','?')}: {f.get('why','')}")
    print(f"      ↳ {f.get('fix','')}")

> 💬 **Discuss (1 min):** the `== np.nan` bug means the data-cleaning branch *never runs* — yet the script trains and prints an accuracy. Would a code review catch it? Would your CI? What does that say about where ML technical debt hides?

## 1.8 · ✍️ Exercise 1 (10 min)

The auditor is blind to **magic numbers** unless a linter happens to mention them. Give it a *fourth tool*: a tiny AST-based detector.

Fill in the `TODO` below, add the tool to the auditor, and re-run the audit. Does a "magic numbers" finding appear?

In [ ]:
import ast

def find_magic_numbers(path: str) -> str:
    """Report numeric literals that are not 0, 1, or -1 (with line numbers)."""
    tree = ast.parse(open(path).read())
    hits = []
    for node in ast.walk(tree):
        # TODO 1: check `isinstance(node, ast.Constant)` and that node.value
        #         is an int/float not in {0, 1, -1}
        # TODO 2: append f"L{node.lineno}: magic number {node.value}" to hits
        pass
    return "\n".join(hits) if hits else "no magic numbers found"

# TODO 3: register the tool and re-audit
# auditor.tools["magic_numbers"] = find_magic_numbers
# findings = audit("inventory.py")
# findings

print(find_magic_numbers("inventory.py"))

<details><summary>💡 Click for solution</summary>

```python
def find_magic_numbers(path: str) -> str:
    tree = ast.parse(open(path).read())
    hits = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
                and node.value not in (0, 1, -1):
            hits.append(f"L{node.lineno}: magic number {node.value}")
    return "\n".join(hits) if hits else "no magic numbers found"

auditor.tools["magic_numbers"] = find_magic_numbers
findings = audit("inventory.py")
```
</details>

**⭐ Bonus (if you're fast):** ask the auditor to also propose a *priority order* for fixing the findings — just change the prompt. Which prompt wording works best with a 1.5B model?

## 1.9 · Checkpoint ✅

Before Part 2, make sure you can answer:

1. Why do we give the LLM *tool evidence* instead of asking it to find smells from raw code alone?
2. What are the four ingredients of an agent?
3. Why did we write tests **before** building any agent that modifies code?

<details><summary>Answers</summary>

1. Tools are deterministic and cheap; grounding the LLM in measured evidence slashes hallucinated findings and lets a small model punch above its weight.
2. Role (system prompt) · brain (LLM) · tools (callables) · output contract (JSON schema).
3. Tests are the behavioural contract — in Part 2 the QA agent uses them to *verify* refactorings. Without them, "improvement" is unverifiable.
</details>

**Save your work:** the next cell stores the findings so Part 2 can pick them up (Part 2 also works standalone if you restart your runtime).

In [ ]:
with open("findings.json", "w") as fh:
    json.dump(findings, fh, indent=2)
print("findings.json saved →", [f["smell"] for f in findings])
print("\n➡️  Continue to Part 2: The Refactoring Team")